# **FEATURE ENGINEERING**

**Pipeline**: Raw CSV → Cleaning (notebook 01) → **Feature Engineering** → EDA → Analytics

Notebook này:
1. Load lại các bảng đã clean từ notebook 01
2. Merge thành 1 master DataFrame
3. Tính toán các derived features cần thiết cho EDA và ML
4. Export `master_df.csv` để các notebook sau dùng

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## **1. Load & Re-clean dữ liệu gốc**

> Chạy lại toàn bộ cleaning logic từ notebook 01 để đảm bảo reproducibility.

In [2]:
# ---------- Load raw files ----------
#read data
sales = pd.read_csv("Dataset/Raw_data/Online_Sales.csv")
customers = pd.read_excel("Dataset/Raw_data/CustomersData.xlsx")
tax = pd.read_excel("Dataset/Raw_data/Tax_amount.xlsx")
discount = pd.read_csv("Dataset/Raw_data/Discount_Coupon.csv")
marketing = pd.read_csv("Dataset/Raw_data/Marketing_Spend.csv")
print("Raw shapes:")
for name, df in [("sales",sales),("customers",customers),("tax",tax),("discount",discount),("marketing",marketing)]:
    print(f"  {name}: {df.shape}")

Raw shapes:
  sales: (52924, 10)
  customers: (1468, 4)
  tax: (20, 2)
  discount: (204, 4)
  marketing: (365, 3)


In [3]:
# =============================================
# CLEANING — SALES
# =============================================
sales["CustomerID"]    = sales["CustomerID"].astype(str)
sales["Transaction_ID"] = sales["Transaction_ID"].astype(str)
sales["Transaction_Date"] = pd.to_datetime(sales["Transaction_Date"], errors="coerce")

for col in ["Product_SKU", "Product_Description", "Product_Category", "Coupon_Status"]:
    sales[col] = sales[col].astype(str).str.strip()

for col in ["Quantity", "Avg_Price", "Delivery_Charges"]:
    sales[col] = pd.to_numeric(sales[col], errors="coerce")

sales = sales.drop_duplicates(subset=["CustomerID", "Transaction_ID", "Product_SKU"])
sales = sales[(sales["Quantity"] > 0) & (sales["Avg_Price"] > 0) & (sales["Delivery_Charges"] >= 0)]
sales = sales.dropna(subset=["CustomerID", "Transaction_ID", "Transaction_Date", "Product_SKU"])

sales["Product_Description"] = sales["Product_Description"].fillna("Unknown")
sales["Coupon_Status"]       = sales["Coupon_Status"].fillna("Not Used").str.title()

# Base revenue (trước tax/discount)
sales["Revenue"] = sales["Quantity"] * sales["Avg_Price"]
sales["Month"]   = sales["Transaction_Date"].dt.strftime("%b")

print(f"Sales clean: {sales.shape}")

Sales clean: (52924, 12)


In [4]:
# =============================================
# CLEANING — CUSTOMERS
# =============================================
customers["CustomerID"] = customers["CustomerID"].astype(str)

for col in ["Gender", "Location"]:
    customers[col] = customers[col].astype(str).str.strip().str.title()

customers = customers[customers["Tenure_Months"] >= 0]
print(f"Customers clean: {customers.shape}")

Customers clean: (1468, 4)


In [5]:
# =============================================
# CLEANING — DISCOUNT
# =============================================
for col in ["Month", "Product_Category", "Coupon_Code"]:
    discount[col] = discount[col].astype(str).str.strip()

discount["Discount_pct"] = pd.to_numeric(discount["Discount_pct"], errors="coerce").fillna(0)
# Normalize: nếu lưu dạng 10 thì / 100 để ra 0.10
if discount["Discount_pct"].max() > 1:
    discount["Discount_pct"] = discount["Discount_pct"] / 100

discount = discount[(discount["Discount_pct"] >= 0) & (discount["Discount_pct"] <= 1)]
print(f"Discount clean: {discount.shape}")

Discount clean: (204, 4)


In [6]:
# =============================================
# CLEANING — TAX
# =============================================
tax["Product_Category"] = tax["Product_Category"].astype(str).str.strip()
tax = tax[(tax["GST"] >= 0) & (tax["GST"] <= 1)]
print(f"Tax clean: {tax.shape}")

Tax clean: (20, 2)


In [7]:
# =============================================
# CLEANING — MARKETING
# =============================================
marketing["Date"] = pd.to_datetime(marketing["Date"], errors="coerce")
marketing = marketing[(marketing["Offline_Spend"] >= 0) & (marketing["Online_Spend"] >= 0)]
marketing["Total_Marketing_Spend"] = marketing["Offline_Spend"] + marketing["Online_Spend"]
print(f"Marketing clean: {marketing.shape}")

Marketing clean: (365, 4)


## **2. Merge thành Master DataFrame**

Strategy:
- `sales` LEFT JOIN `customers` (thêm Gender, Location, Tenure)
- LEFT JOIN `tax` (thêm GST rate)
- LEFT JOIN `discount` on Month + Category (thêm Discount_pct)
- LEFT JOIN `marketing` on Date (thêm daily marketing spend)

In [8]:
# Step 1: sales + customers
df = sales.merge(customers, on="CustomerID", how="left")
print(f"After +customers: {df.shape}")

# Step 2: + tax
df = df.merge(tax, on="Product_Category", how="left")
print(f"After +tax: {df.shape}")

# Step 3: + discount (join on Month + Category)
df = df.merge(
    discount[["Month", "Product_Category", "Discount_pct"]],
    on=["Month", "Product_Category"],
    how="left"
)
# Nếu không có discount, fill 0
df["Discount_pct"] = df["Discount_pct"].fillna(0)
# Nếu Coupon_Status != Used thì discount = 0 (coupon có nhưng không dùng)
df.loc[df["Coupon_Status"] != "Used", "Discount_pct"] = 0
print(f"After +discount: {df.shape}")

# Step 4: + marketing (join on date)
marketing_daily = marketing[["Date", "Offline_Spend", "Online_Spend", "Total_Marketing_Spend"]].copy()
marketing_daily = marketing_daily.rename(columns={"Date": "Transaction_Date"})
df = df.merge(marketing_daily, on="Transaction_Date", how="left")
print(f"After +marketing: {df.shape}")

After +customers: (52924, 15)
After +tax: (52924, 16)
After +discount: (52924, 17)
After +marketing: (52924, 20)


In [9]:
df.head()

,CustomerID,Transaction_ID,Transaction_Date,Product_SKU,Product_Description,Product_Category,Quantity,Avg_Price,Delivery_Charges,Coupon_Status,Revenue,Month,Gender,Location,Tenure_Months,GST,Discount_pct,Offline_Spend,Online_Spend,Total_Marketing_Spend
0,17850,16679,2019-01-01,GGOENEBJ079499,Nest Learning Thermostat 3rd Gen-USA - Stainle...,Nest-USA,1,153.71,6.50,Used,153.71,Jan,M,Chicago,12,0.10,0.10,4500,2424.50,6924.50
1,17850,16680,2019-01-01,GGOENEBJ079499,Nest Learning Thermostat 3rd Gen-USA - Stainle...,Nest-USA,1,153.71,6.50,Used,153.71,Jan,M,Chicago,12,0.10,0.10,4500,2424.50,6924.50
2,17850,16681,2019-01-01,GGOEGFKQ020399,Google Laptop and Cell Phone Stickers,Office,1,2.05,6.50,Used,2.05,Jan,M,Chicago,12,0.10,0.10,4500,2424.50,6924.50
3,17850,16682,2019-01-01,GGOEGAAB010516,Google Men's 100% Cotton Short Sleeve Hero Tee...,Apparel,5,17.53,6.50,Not Used,87.65,Jan,M,Chicago,12,0.18,0.00,4500,2424.50,6924.50
4,17850,16682,2019-01-01,GGOEGBJL013999,Google Canvas Tote Natural/Navy,Bags,1,16.50,6.50,Used,16.50,Jan,M,Chicago,12,0.18,0.10,4500,2424.50,6924.50


In [10]:
# Kiểm tra tỷ lệ null sau merge
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
print("Null % sau merge:")
print(null_pct[null_pct > 0])

Null % sau merge:
Series([], dtype: float64)


## **3. Feature Engineering**

Tạo các features phục vụ:
- **EDA**: revenue metrics, time features
- **RFM Segmentation**: Recency, Frequency, Monetary
- **Churn Prediction**: behavioral signals
- **CLV**: historical spend patterns
- **Recommendation**: purchase history

### 3.1 Time Features

**Muc dich**: Trich xuat cac chieu thoi gian tu `Transaction_Date` de phuc vu phan tich theo chu ky (thang, tuan).

| Feature | Y nghia |
|---|---|
| `Month` | Ten thang (Jan, Feb...) — dung cho seasonality analysis va join voi bang discount |
| `Date` | Ngay thuan (khong gio) — dung khi group theo ngay |
| `Week` | Tuan theo ISO period — dung cho weekly trend |

In [11]:
# Chuan hoa thoi gian & ID
df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"], errors="coerce")
df["Month"]            = df["Transaction_Date"].dt.strftime("%b")
df["Date"]             = df["Transaction_Date"].dt.date
df["Week"]             = df["Transaction_Date"].dt.to_period("W").astype(str)

df["CustomerID"]     = df["CustomerID"].astype(str)
df["Transaction_ID"] = df["Transaction_ID"].astype(str)

### 3.2 Coupon / Discount Flags

**Muc dich**: Tao co nhi phan de phan tich hanh vi dung coupon va anh huong cua chiet khau den doanh thu.

| Feature | Y nghia |
|---|---|
| `Is_Discounted` | 1 neu giao dich co ap dung chiet khau (Discount_pct > 0), 0 neu khong |
| `Is_Used_Coupon` | 1 neu khach hang thuc su dung coupon (`Coupon_Status == "Used"`), 0 neu khong |

> **Luu y**: Hai co nay co the khac nhau — coupon duoc cap nhung chua chac da dung.

In [12]:
# Co dung coupon / co giam gia
df["Is_Discounted"]  = (df["Discount_pct"] > 0).astype(int)
df["Is_Used_Coupon"] = (df["Coupon_Status"] == "Used").astype(int)

### 3.3 Invoice — Gia tri hoa don thuc te

**Muc dich**: Tinh so tien cuoi cung khach hang thanh toan, bao gom day du: so luong, don gia, chiet khau (neu dung coupon), thue GST va phi van chuyen.

**Cong thuc**:

Neu dung coupon:
Invoice = (Quantity x Avg_Price x (1 - Discount_pct) x (1 + GST)) + Delivery_Charges

Neu khong dung coupon:
Invoice = (Quantity x Avg_Price x (1 + GST)) + Delivery_Charges

In [13]:
# Discount_pct da o dang thap phan (0.10), khong chia them /100
df["Invoice"] = np.where(
    df["Coupon_Status"] == "Used",
    (df["Quantity"] * df["Avg_Price"]) * (1 - df["Discount_pct"]) * (1 + df["GST"]) + df["Delivery_Charges"],
    (df["Quantity"] * df["Avg_Price"]) * (1 + df["GST"]) + df["Delivery_Charges"]
)

print("Invoice stats:")
print(df["Invoice"].describe().round(2))

Invoice stats:
count   52924.00
mean      101.98
std       172.37
min         4.60
25%        20.16
50%        45.64
75%       137.40
max      8979.28
Name: Invoice, dtype: float64


### 3.4 Total Revenue — Doanh thu gop (dung cho ABC)

**Muc dich**: Tinh doanh thu gop o cap giao dich de phuc vu phan loai ABC theo danh muc san pham.

**Cong thuc** (nhat quan voi `Invoice`, bo phi van chuyen vi day la chi phi logistics):


In [14]:
# Bo sung GST vao total_revenue de nhat quan voi Invoice
df["total_revenue"] = (
    df["Quantity"] * df["Avg_Price"]
    * (1 - df["Discount_pct"])
    * (1 + df["GST"])
)

### 3.5 ABC Category — Phan loai san pham theo quy tac Pareto

**Muc dich**: Ap dung nguyen tac Pareto (80/20) de phan nhom danh muc san pham theo dong gop doanh thu.

| Nhom | Ty le doanh thu tich luy | Y nghia chien luoc |
|---|---|---|
| **A** | 0 - 80% | Cash cow — uu tien ton kho, marketing, khong giam gia sau |
| **B** | 80 - 95% | Tiem nang — theo doi, co the day len nhom A |
| **C** | 95 - 100% | Dai duoi — xem xet loai bo hoac bundle |

In [15]:
def abc_categorizer(df):
    grouped_df = (
        df.groupby("Product_Category", as_index=False)["total_revenue"]
          .sum()
          .rename(columns={"total_revenue": "cat_revenue"})
    )
    grouped_df = grouped_df.sort_values("cat_revenue", ascending=False)

    total_rev = grouped_df["cat_revenue"].sum()
    grouped_df["share"]     = grouped_df["cat_revenue"] / total_rev
    grouped_df["cum_share"] = grouped_df["share"].cumsum()

    def assign_class(cum):
        if cum <= 0.80:
            return "A"
        elif cum <= 0.95:
            return "B"
        else:
            return "C"

    grouped_df["ABC"] = grouped_df["cum_share"].apply(assign_class)
    print(grouped_df[["Product_Category", "cat_revenue", "cum_share", "ABC"]].to_string(index=False))
    return grouped_df.set_index("Product_Category")["ABC"].to_dict()


abc_dict  = abc_categorizer(df)
df["ABC"] = df["Product_Category"].map(abc_dict)
print("\nABC distribution:\n", df["ABC"].value_counts())

    Product_Category  cat_revenue  cum_share ABC
            Nest-USA   2622005.70       0.54   A
             Apparel    650964.77       0.68   A
                Nest    504570.87       0.78   A
              Office    283497.36       0.84   B
           Drinkware    222493.78       0.88   B
                Bags    167981.56       0.92   B
Notebooks & Journals    106610.16       0.94   B
           Lifestyle     82039.84       0.96   C
         Nest-Canada     73342.81       0.97   C
            Headgear     50397.95       0.98   C
          Gift Cards     19447.44       0.99   C
              Google     10362.52       0.99   C
           Backpacks      9649.96       0.99   C
         Accessories      7412.02       0.99   C
                 Fun      7114.23       1.00   C
                Waze      6952.93       1.00   C
             Bottles      6667.34       1.00   C
          Housewares      4985.39       1.00   C
           More Bags      3477.41       1.00   C
             Android

### 3.6 RFM — Recency, Frequency, Monetary

**Muc dich**: Phan khuc khach hang theo hanh vi mua sam. RFM la nen tang cho phan tich churn, CLV va ca nhan hoa marketing.

| Metric | Dinh nghia | Dien giai |
|---|---|---|
| **Recency** | So ngay ke tu lan mua cuoi | Cang nho = khach cang 'nong' |
| **Frequency** | So transaction ID duy nhat | Cang lon = khach cang trung thanh |
| **Monetary** | Tong Invoice da thanh toan | Cang lon = khach cang gia tri |

**Scoring**:
- `R_Score`: 1 (lau nhat) den 4 (gan nhat) — **dao chieu** so voi gia tri Recency
- `F_Score`, `M_Score`: 1 (thap nhat) den 4 (cao nhat)
- `RFM_Score` = R + F + M (tong don gian, toi da = 12)

> **BUG DA SUA**:  
> 1. Code cu khong dao chieu R — Recency cao (mua lau roi) van duoc R_Score cao, nguoc voi dinh nghia RFM.  
> 2. Code cu nhan tat ca scores voi 10 truoc khi cong (`R*10 + F*10 + M*10`), khien thang diem sai.  
> 3. Thay hardcode nguong bang `pd.qcut` — phan phoi deu hon tren moi dataset.

In [16]:
# Snapshot ngay hien tai
today = df["Transaction_Date"].max() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerID").agg(
    Recency   = ("Transaction_Date", lambda x: (today - x.max()).days),
    Frequency = ("Transaction_ID", "nunique"),
    Monetary  = ("Invoice", "sum")
).reset_index()

# FIX 1: R_Score dao chieu (Recency nho = tot = score cao)
# FIX 2: Dung qcut de chia deu theo phan phoi thuc te cua data
rfm["R_Score"] = pd.qcut(rfm["Recency"],  q=4, labels=[4, 3, 2, 1]).astype(int)
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), q=4, labels=[1, 2, 3, 4]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary"].rank(method="first"),  q=4, labels=[1, 2, 3, 4]).astype(int)

# FIX 3: RFM_Score la tong don gian (3-12), khong nhan 10
rfm["RFM_Score"] = rfm["R_Score"] + rfm["F_Score"] + rfm["M_Score"]

def heuristic_segment(score):
    if score <= 5:
        return "Standard"    # 3-5
    elif score <= 8:
        return "Silver"      # 6-8
    elif score <= 10:
        return "Premium"     # 9-10
    else:
        return "Gold"        # 11-12

rfm["Heuristic_Segment"] = rfm["RFM_Score"].apply(heuristic_segment)

print("RFM stats:")
print(rfm[["Recency","Frequency","Monetary"]].describe().round(2))
print("\nSegment distribution:\n", rfm["Heuristic_Segment"].value_counts())

RFM stats:
       Recency  Frequency  Monetary
count  1468.00    1468.00   1468.00
mean    145.29      18.14   3676.67
std     101.94      24.98   5846.08
min       1.00       1.00      6.99
25%      56.00       5.00    783.97
50%     132.00      11.00   2011.62
75%     221.00      23.00   4495.06
max     365.00     328.00  87200.90

Segment distribution:
 Heuristic_Segment
Silver      494
Standard    407
Premium     315
Gold        252
Name: count, dtype: int64


In [17]:
# Gan RFM & segment vao df
df = df.merge(
    rfm[[
        "CustomerID", "Recency", "Frequency", "Monetary",
        "R_Score", "F_Score", "M_Score", "RFM_Score", "Heuristic_Segment"
    ]],
    on="CustomerID",
    how="left"
)

### 3.7 Marketing Monthly — Bang tong hop chi phi marketing theo thang

**Muc dich**: Tong hop doanh thu va chi tieu marketing theo thang de phan tich ROI.

| Feature | Y nghia |
|---|---|
| `Invoice` | Tong doanh thu thuc te trong thang |
| `Total_Mkt_Spend` | Tong chi phi marketing (online + offline) trong thang |
| `Pct_Mkt_Spend` | Ty le chi phi marketing / doanh thu (%) — do luong hieu qua dau tu |
| `Pct_Delivery` | Ty le phi van chuyen / doanh thu (%) — do luong ganh nang logistics |

> **Luu y**: Bang nay la bang phan tich rieng (aggregated), khong merge vao `df` chinh.

In [18]:
marketing_monthly = (
    df.groupby("Month", as_index=False)
      .agg(
          Invoice         = ("Invoice", "sum"),
          Discount_pct    = ("Discount_pct", "mean"),
          GST             = ("GST", "mean"),
          DeliveryCharges = ("Delivery_Charges", "sum"),
          Total_Mkt_Spend = ("Total_Marketing_Spend", "sum")
      )
)

marketing_monthly["Pct_Mkt_Spend"] = (
    marketing_monthly["Total_Mkt_Spend"] * 100 / marketing_monthly["Invoice"]
)
marketing_monthly["Pct_Delivery"] = (
    marketing_monthly["DeliveryCharges"] * 100 / marketing_monthly["Invoice"]
)

print(marketing_monthly.round(2))

   Month   Invoice  Discount_pct  GST  DeliveryCharges  Total_Mkt_Spend  \
0    Apr 477498.59          0.03 0.14         41481.74      21655922.13   
1    Aug 475796.88          0.07 0.15         61099.57      28385733.77   
2    Dec 556112.29          0.10 0.12         37881.99      28964402.01   
3    Feb 375162.05          0.07 0.14         49216.60      15841536.05   
4    Jan 494090.55          0.03 0.13         59242.32      20052775.17   
5    Jul 451878.41          0.03 0.14         48723.93      20618934.41   
6    Jun 361000.17          0.10 0.14         37513.58      18625403.73   
7    Mar 415157.79          0.10 0.14         60799.94      17453780.31   
8    May 365596.03          0.06 0.14         41396.17      17525521.02   
9    Nov 547788.13          0.07 0.12         32311.93      21096299.69   
10   Oct 480767.37          0.03 0.13         45961.88      20536272.39   
11   Sep 396510.49          0.10 0.14         41005.42      19257626.34   

    Pct_Mkt_Spend  Pct_D

### 3.8 K-Means Segmentation — Phan cum khach hang (tuy chon)

**Muc dich**: Dung thuat toan K-Means de phan cum khach hang dua tren RFM, bo sung cho heuristic segment.

| Feature | Y nghia |
|---|---|
| `KMeans_Label` | Nhan cum (0,1,2,3) — can dien giai them sau khi xem centroid |

**Tai sao can StandardScaler?** Recency, Frequency, Monetary co don vi va bien do rat khac nhau. Neu khong scale, K-Means bi dominate boi Monetary va bo qua Frequency.

> **Luu y**: `k=4` la lua chon mac dinh. Nen kiem tra bang Elbow Method hoac Silhouette Score truoc.

In [19]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

kmeans_features = rfm[["Recency", "Frequency", "Monetary"]].copy()
scaler        = StandardScaler()
kmeans_scaled = scaler.fit_transform(kmeans_features)

k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
rfm["KMeans_Label"] = kmeans.fit_predict(kmeans_scaled)

df = df.merge(rfm[["CustomerID", "KMeans_Label"]], on="CustomerID", how="left")

print("KMeans cluster profile:")
print(rfm.groupby("KMeans_Label")[["Recency","Frequency","Monetary"]].mean().round(2))

KMeans cluster profile:
              Recency  Frequency  Monetary
KMeans_Label                              
0               80.26      55.69  11932.70
1              255.17      10.29   1969.64
2               79.25      12.79   2433.38
3               81.83     270.67  65739.45


---
## **4. Export master_df.csv**

Export **một lần duy nhất** tại đây với đầy đủ tất cả các cột đã tính.  
EDA và các notebook tiếp theo **chỉ cần `pd.read_csv("Output/master_df.csv")`** 

| Nhóm cột | Cột |
|---|---|
| Identity | `CustomerID`, `Transaction_ID`, `Transaction_Date` |
| Time | `Month`, `Month_Num`, `Year`, `YearMonth`, `Date`, `Week`, `DayName`, `Season` |
| Product | `Product_SKU`, `Product_Description`, `Product_Category`, `ABC` |
| Financial | `Quantity`, `Avg_Price`, `Delivery_Charges`, `Revenue`, `total_revenue`, `Invoice`, `Discount_pct`, `Discount_Amount`, `GST` |
| Customer | `Gender`, `Location`, `Tenure_Months` |
| Coupon | `Coupon_Status`, `Is_Discounted`, `Is_Used_Coupon`, `Coupon_Used` |
| Marketing | `Offline_Spend`, `Online_Spend`, `Total_Marketing_Spend` |
| RFM | `Recency`, `Frequency`, `Monetary`, `R_Score`, `F_Score`, `M_Score`, `RFM_Score`, `Heuristic_Segment`, `KMeans_Label` |


In [20]:
# ── 4. Export master_df.csv ──────────────────────────────────────────────────
import os
os.makedirs("Output", exist_ok=True)

# Danh sách cột muốn export (theo thứ tự logic)
export_cols = [
    # Identity
    "CustomerID", "Transaction_ID", "Transaction_Date",
    # Time — đầy đủ để EDA dùng trực tiếp
    "Month", "Month_Num", "Year", "YearMonth", "Date", "Week", "DayName", "Season",
    # Product
    "Product_SKU", "Product_Description", "Product_Category", "ABC",
    # Financial
    "Quantity", "Avg_Price", "Delivery_Charges",
    "Revenue",          # Qty × Price (pre-tax, pre-discount) — raw
    "total_revenue",    # Qty × Price × (1-discount) × (1+GST) — basis ABC
    "Invoice",          # total_revenue + Delivery — tiền khách thực trả
    "Discount_pct", "Discount_Amount", "GST",
    # Customer
    "Gender", "Location", "Tenure_Months",
    # Coupon
    "Coupon_Status", "Is_Discounted", "Is_Used_Coupon", "Coupon_Used",
    # Marketing
    "Offline_Spend", "Online_Spend", "Total_Marketing_Spend",
    # RFM
    "Recency", "Frequency", "Monetary",
    "R_Score", "F_Score", "M_Score", "RFM_Score",
    "Heuristic_Segment", "KMeans_Label",
]

# Chỉ export cột thực sự tồn tại (tránh KeyError nếu KMeans bị skip)
actual_cols = [c for c in export_cols if c in df.columns]
missing_export = [c for c in export_cols if c not in df.columns]
if missing_export:
    print(f"⚠️  Cột chưa có (sẽ không export): {missing_export}")

master_df = df[actual_cols].copy()
master_df.to_csv("Output/master_df.csv", index=False)

print(f"✅ Exported master_df.csv: {master_df.shape[0]:,} rows × {master_df.shape[1]} cols")
print(f"   Path: Output/master_df.csv")
print()
print("── Cột đã export ──")
for g, cols in [
    ("Identity  ", ["CustomerID","Transaction_ID","Transaction_Date"]),
    ("Time      ", ["Month","Month_Num","Year","YearMonth","Date","Week","DayName","Season"]),
    ("Product   ", ["Product_SKU","Product_Description","Product_Category","ABC"]),
    ("Financial ", ["Revenue","total_revenue","Invoice","Discount_pct","Discount_Amount","GST","Delivery_Charges"]),
    ("Customer  ", ["Gender","Location","Tenure_Months"]),
    ("Coupon    ", ["Coupon_Status","Is_Discounted","Is_Used_Coupon","Coupon_Used"]),
    ("Marketing ", ["Offline_Spend","Online_Spend","Total_Marketing_Spend"]),
    ("RFM       ", ["Recency","Frequency","Monetary","R_Score","F_Score","M_Score","RFM_Score","Heuristic_Segment","KMeans_Label"]),
]:
    present = [c for c in cols if c in actual_cols]
    print(f"  {g}: {present}")


⚠️  Cột chưa có (sẽ không export): ['Month_Num', 'Year', 'YearMonth', 'DayName', 'Season', 'Discount_Amount', 'Coupon_Used']
✅ Exported master_df.csv: 52,924 rows × 36 cols
   Path: Output/master_df.csv

── Cột đã export ──
  Identity  : ['CustomerID', 'Transaction_ID', 'Transaction_Date']
  Time      : ['Month', 'Date', 'Week']
  Product   : ['Product_SKU', 'Product_Description', 'Product_Category', 'ABC']
  Financial : ['Revenue', 'total_revenue', 'Invoice', 'Discount_pct', 'GST', 'Delivery_Charges']
  Customer  : ['Gender', 'Location', 'Tenure_Months']
  Coupon    : ['Coupon_Status', 'Is_Discounted', 'Is_Used_Coupon']
  Marketing : ['Offline_Spend', 'Online_Spend', 'Total_Marketing_Spend']
  RFM       : ['Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score', 'Heuristic_Segment', 'KMeans_Label']
